# Fraud Detection Rules Discovery

## Using Decision Trees and Association Rule Mining

This notebook discovers interpretable fraud detection rules from clustering insights using:
- Decision Tree Classification
- Association Rule Mining (Apriori)
- Rule extraction and evaluation

In [ ]:
# Import libraries for rule discovery
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported for rule discovery")

## Part 1: Decision Tree for Fraud Detection Rules

In [ ]:
# Decision Tree for Fraud Detection Rules
print("=== DECISION TREE ANALYSIS FOR FRAUD DETECTION ===\n")

# Prepare data for decision tree
X_tree = X.copy()  # Features
y_tree = y.copy()  # Target (fraud flag)

# Train decision tree with limited depth for interpretability
print("Training Decision Tree Classifier...")
dt_classifier = DecisionTreeClassifier(
    max_depth=5,  # Limited depth for interpretable rules
    min_samples_split=100,  # Minimum samples to split
    min_samples_leaf=50,    # Minimum samples in leaf
    random_state=42
)

dt_classifier.fit(X_tree, y_tree)

print(f"✅ Decision Tree trained!")
print(f"   • Tree depth: {dt_classifier.get_depth()}")
print(f"   • Number of leaves: {dt_classifier.get_n_leaves()}")
print(f"   • Feature importance: {X_tree.shape[1]} features considered")

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X_tree.columns,
    'importance': dt_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n=== TOP 20 MOST IMPORTANT FEATURES FOR FRAUD DETECTION ===")
print(feature_importance.head(20).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 20 Features for Fraud Detection (Decision Tree)')
plt.tight_layout()
plt.show()

print("\n✅ Feature importance visualization created")

In [ ]:
# Decision Tree Performance
print("\n=== DECISION TREE PERFORMANCE ===")

y_pred_tree = dt_classifier.predict(X_tree)
y_pred_proba = dt_classifier.predict_proba(X_tree)[:, 1]

print(f"\nClassification Report:")
print(classification_report(y_tree, y_pred_tree, target_names=['Legitimate', 'Fraud']))

# AUC-ROC Score
auc_score = roc_auc_score(y_tree, y_pred_proba)
print(f"\nAUC-ROC Score: {auc_score:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_tree, y_pred_tree)
print(f"\nConfusion Matrix:")
print(f"                Predicted Legit  Predicted Fraud")
print(f"Actual Legit         {cm[0,0]:>6}          {cm[0,1]:>6}")
print(f"Actual Fraud         {cm[1,0]:>6}          {cm[1,1]:>6}")

In [ ]:
# Extract Decision Tree Rules
print("\n=== DECISION TREE RULES (TEXT FORMAT) ===\n")

tree_rules_text = export_text(dt_classifier, feature_names=list(X_tree.columns))
print(tree_rules_text)

In [ ]:
# Visualize Decision Tree
print("\n=== VISUALIZING DECISION TREE ===\n")

plt.figure(figsize=(25, 15))
plot_tree(dt_classifier, 
          feature_names=list(X_tree.columns),
          class_names=['Legitimate', 'Fraud'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree for Fraud Detection', fontsize=20)
plt.tight_layout()
plt.show()

print("✅ Decision Tree visualization created")

## Part 2: Extract Interpretable Decision Rules from Tree

In [ ]:
# Extract interpretable rules from decision tree
print("=== EXTRACTING INTERPRETABLE FRAUD DETECTION RULES ===\n")

def get_tree_rules(tree, feature_names):
    """Extract decision rules from tree nodes."""
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined!"
        for i in tree_.feature
    ]
    
    rules = []
    
    def recurse(node, depth, conditions, is_fraud_side):
        indent = "  " * depth
        if tree_.feature[node] != -2:  # Not a leaf
            name = feature_name[node]
            threshold = tree_.threshold[node]
            
            # Left child (<=)
            left_conditions = conditions + [f"{name} <= {threshold:.4f}"]
            recurse(tree_.children_left[node], depth + 1, left_conditions, is_fraud_side)
            
            # Right child (>)
            right_conditions = conditions + [f"{name} > {threshold:.4f}"]
            recurse(tree_.children_right[node], depth + 1, right_conditions, is_fraud_side)
        else:  # Leaf node
            # Get samples and value at this leaf
            samples = tree_.n_node_samples[node]
            value = tree_.value[node][0]
            fraud_count = int(value[1])
            legit_count = int(value[0])
            
            if fraud_count + legit_count > 0:
                fraud_rate = fraud_count / (fraud_count + legit_count)
                rules.append({
                    'conditions': ' AND '.join(conditions),
                    'fraud_count': fraud_count,
                    'legit_count': legit_count,
                    'total_samples': samples,
                    'fraud_rate': fraud_rate,
                    'confidence': fraud_rate
                })
        
    recurse(0, 1, [], False)
    return rules

# Get all rules
all_rules = get_tree_rules(dt_classifier, list(X_tree.columns))

# Convert to DataFrame
rules_df = pd.DataFrame(all_rules).sort_values('fraud_rate', ascending=False)

print(f"\nTotal rules extracted: {len(rules_df)}\n")

# Filter high-confidence fraud rules (fraud_rate > 50%)
fraud_rules = rules_df[rules_df['fraud_rate'] > 0.5]

print(f"=== HIGH-CONFIDENCE FRAUD RULES (Fraud Rate > 50%) ===")
print(f"Found {len(fraud_rules)} high-confidence fraud detection rules\n")

for idx, (rule_idx, rule) in enumerate(fraud_rules.head(10).iterrows(), 1):
    print(f"\n--- Rule #{idx} ---")
    print(f"Condition: {rule['conditions']}")
    print(f"Fraud Rate (Confidence): {rule['fraud_rate']:.1%}")
    print(f"Fraud Cases: {rule['fraud_count']:.0f}")
    print(f"Legitimate Cases: {rule['legit_count']:.0f}")
    print(f"Total Cases Matching Rule: {rule['total_samples']:.0f}")
    print(f"Support (% of total): {rule['total_samples'] / len(X_tree) * 100:.2f}%")

In [ ]:
# Filter legitimate rules (fraud_rate < 20%)
print("\n\n=== LOW-FRAUD RULES (Fraud Rate < 20%) ===")

legit_rules = rules_df[rules_df['fraud_rate'] < 0.2].sort_values('total_samples', ascending=False)
print(f"Found {len(legit_rules)} low-fraud detection rules\n")

for idx, (rule_idx, rule) in enumerate(legit_rules.head(10).iterrows(), 1):
    print(f"\n--- Legitimate Rule #{idx} ---")
    print(f"Condition: {rule['conditions']}")
    print(f"Fraud Rate: {rule['fraud_rate']:.1%}")
    print(f"Fraud Cases: {rule['fraud_count']:.0f}")
    print(f"Legitimate Cases: {rule['legit_count']:.0f}")
    print(f"Total Cases Matching Rule: {rule['total_samples']:.0f}")
    print(f"Support (% of total): {rule['total_samples'] / len(X_tree) * 100:.2f}%")

In [ ]:
# Save rules to CSV
rules_df_export = rules_df.copy()
rules_df_export['support'] = rules_df_export['total_samples'] / len(X_tree)
rules_df_export = rules_df_export[['conditions', 'fraud_rate', 'confidence', 'fraud_count', 'legit_count', 'total_samples', 'support']]
rules_df_export.columns = ['Rule Conditions', 'Fraud Rate', 'Confidence', 'Fraud Count', 'Legitimate Count', 'Total Samples', 'Support']

rules_csv_path = 'decision_tree_fraud_rules.csv'
rules_df_export.to_csv(rules_csv_path, index=False)
print(f"✅ Decision tree rules saved to: {rules_csv_path}")

## Part 3: Association Rule Mining (Apriori)

In [ ]:
# Prepare data for association rule mining
print("\n=== ASSOCIATION RULE MINING FOR FRAUD PATTERNS ===\n")

print("Preparing data for association rule mining...\n")

# Discretize numerical features into bins for association rules
X_discrete = X.copy()

# Create categorical features from numerical ones
for col in X_discrete.columns:
    # Use quartile-based binning
    X_discrete[col] = pd.qcut(X[col], q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'], duplicates='drop')

# Add target variable
X_discrete['fraud_flag'] = y

print(f"Discretized data shape: {X_discrete.shape}")
print(f"Sample of discretized features:")
print(X_discrete.head(10))

In [ ]:
# Create transaction database for association rules
print("\n=== CREATING TRANSACTION DATABASE ===\n")

# Create itemsets (feature-value pairs)
def create_transactions(df):
    transactions = []
    for idx, row in df.iterrows():
        transaction = []
        for col, val in row.items():
            transaction.append(f"{col}={val}")
        transactions.append(transaction)
    return transactions

transactions = create_transactions(X_discrete)
print(f"Total transactions: {len(transactions)}")
print(f"Sample transaction: {transactions[0]}")

# Create one-hot encoded dataframe
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nEncoded dataframe shape: {df_encoded.shape}")
print(f"Sample of encoded itemsets:")
print(df_encoded.head())

In [ ]:
# Apply Apriori algorithm
print("\n=== APPLYING APRIORI ALGORITHM ===\n")

print("Finding frequent itemsets (min_support=0.01)...")
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

print(f"\nFrequent itemsets found: {len(frequent_itemsets)}")
print(f"\nTop 20 most frequent itemsets:")
print(frequent_itemsets.nlargest(20, 'support')[['support', 'itemsets']])

In [ ]:
# Generate association rules
print("\n=== GENERATING ASSOCIATION RULES ===\n")

if len(frequent_itemsets) > 0:
    # Generate rules
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
    
    if len(rules) > 0:
        # Add lift and other metrics
        rules['lift'] = rules['lift'].fillna(0)
        rules['leverage'] = rules['leverage'].fillna(0)
        rules['conviction'] = rules['conviction'].fillna(0)
        
        print(f"Association rules generated: {len(rules)}")
        print(f"\nTop rules sorted by lift (confidence >= 0.5):")
        
        # Show top rules
        top_rules = rules.nlargest(20, 'lift')
        
        for idx, (rule_idx, rule) in enumerate(top_rules.iterrows(), 1):
            antecedent = ', '.join(list(rule['antecedents']))
            consequent = ', '.join(list(rule['consequents']))
            
            print(f"\n--- Rule #{idx} ---")
            print(f"If: {antecedent}")
            print(f"Then: {consequent}")
            print(f"Confidence: {rule['confidence']:.1%}")
            print(f"Support: {rule['support']:.4f}")
            print(f"Lift: {rule['lift']:.2f}")
    else:
        print("No association rules found with the given thresholds.")
else:
    print("No frequent itemsets found.")

In [ ]:
# Focus on fraud-related rules
print("\n=== FRAUD-SPECIFIC ASSOCIATION RULES ===\n")

if len(rules) > 0:
    # Filter rules that imply fraud
    fraud_related = rules[rules['consequents'].apply(lambda x: any('fraud_flag=True' in str(item) for item in x))]
    
    print(f"Rules predicting fraud: {len(fraud_related)}")
    
    if len(fraud_related) > 0:
        # Sort by confidence
        fraud_rules_sorted = fraud_related.nlargest(15, 'confidence')
        
        print(f"\n=== TOP FRAUD PREDICTION RULES (High Confidence) ===")
        
        for idx, (rule_idx, rule) in enumerate(fraud_rules_sorted.iterrows(), 1):
            antecedent = ', '.join(list(rule['antecedents']))
            
            print(f"\n--- Fraud Rule #{idx} ---")
            print(f"Conditions: {antecedent}")
            print(f"Then: FRAUD (confidence: {rule['confidence']:.1%})")
            print(f"Support: {rule['support']:.4f} ({rule['support']*100:.2f}% of data)")
            print(f"Lift: {rule['lift']:.2f}x (likelihood increase vs baseline)")
    else:
        print("No direct fraud prediction rules found.")
else:
    print("No rules available for analysis.")

In [ ]:
# Save association rules
if len(rules) > 0:
    # Convert sets to strings for CSV export
    rules_export = rules.copy()
    rules_export['antecedents'] = rules_export['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules_export['consequents'] = rules_export['consequents'].apply(lambda x: ', '.join(list(x)))
    
    rules_export_csv = rules_export[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False)
    
    assoc_rules_path = 'association_rules_fraud.csv'
    rules_export_csv.to_csv(assoc_rules_path, index=False)
    print(f"✅ Association rules saved to: {assoc_rules_path}")
else:
    print("No rules to save.")

## Part 4: Summary and Actionable Insights

In [ ]:
# Summary of findings
print("\n" + "="*80)
print("FRAUD DETECTION RULES DISCOVERY - SUMMARY")
print("="*80)

print(f"\n1. DECISION TREE ANALYSIS")
print(f"   • Tree depth: {dt_classifier.get_depth()}")
print(f"   • Number of leaves: {dt_classifier.get_n_leaves()}")
print(f"   • AUC-ROC Score: {auc_score:.4f}")
print(f"   • Total rules extracted: {len(rules_df)}")
print(f"   • High-confidence fraud rules (>50%): {len(fraud_rules)}")
print(f"   • Low-fraud rules (<20%): {len(legit_rules)}")

print(f"\n2. TOP 5 MOST IMPORTANT FEATURES")
for idx, (_, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    print(f"   {idx}. {row['feature']}: {row['importance']:.4f}")

if len(rules) > 0:
    print(f"\n3. ASSOCIATION RULE MINING")
    print(f"   • Frequent itemsets: {len(frequent_itemsets)}")
    print(f"   • Association rules: {len(rules)}")
    print(f"   • High-confidence rules (>50%): {len(rules[rules['confidence'] > 0.5])}")
    print(f"   • Fraud prediction rules: {len(fraud_related) if len(rules) > 0 and 'fraud_related' in locals() else 0}")

print(f"\n4. ACTIONABLE INSIGHTS")
print(f"   ✓ Decision tree provides interpretable split rules")
print(f"   ✓ Feature importance identifies key fraud indicators")
print(f"   ✓ Association rules reveal fraud patterns")
print(f"   ✓ Rules can be deployed in fraud detection systems")
print(f"   ✓ Confidence scores guide alert thresholds")

print(f"\n5. FILES GENERATED")
print(f"   • decision_tree_fraud_rules.csv")
print(f"   • association_rules_fraud.csv")

print("\n" + "="*80)